# 05. Distribution Fault Epidemiology, Duration & Reliability Metrics

## Electrical Engineering Context
Distribution protection relays isolate abnormal conditions to prevent conductor burn-downs and transformer thermal degradation:
* **IEEE 50/51:** Instantaneous / Time-delay Overcurrent
* **IEEE 27:** Undervoltage / Bus Sag
* **IEEE 59:** Overvoltage / Ferranti Swell
* **IEEE 46:** Negative-sequence Current Imbalance
* **IEEE 49:** Thermal Overload

In this notebook, we analyze fault frequency, compute a transparent **Fault Severity Score**, evaluate outage downtime, and examine recurring feeder trip intervals.


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

from src.utils.config import PROCESSED_DATA_DIR
from src.fault_analysis.fault_engine import FaultAnalyticsEngine

df_faults = pd.read_csv(PROCESSED_DATA_DIR / "fact_fault_events.csv")
df_faults["timestamp"] = pd.to_datetime(df_faults["timestamp"])
print(f"Loaded {len(df_faults)} historical protection relay fault events.")


## 1. Reliability Indices & Fault Epidemiology Summary
We calculate MTBF, MTTR, and cumulative outage hours.


In [ ]:
summary = FaultAnalyticsEngine.compute_summary(df_faults)
for k, v in summary.items():
    if not isinstance(v, dict):
        print(f"{k.replace('_', ' ').title()}: {v}")


## 2. Fault Severity Score Distribution
Our transparent rule-based score is computed as:
$$\text{Severity} = 35 \times \left(\frac{I_{peak}}{2 \cdot I_{rated}}\right) + 25 \times \left(\frac{\text{Sag \%}}{20}\right) + 20 \times \left(\frac{\text{Duration}}{120}\right) + 20$$


In [ ]:
plt.figure(figsize=(10, 4.5))
sns.histplot(data=df_faults, x="severity_score", hue="fault_type_id", multiple="stack", bins=20, palette="tab10")
plt.title("Distribution of Rule-Based Fault Severity Scores", fontsize=12, fontweight="bold")
plt.xlabel("Severity Score (0 - 100)")
plt.ylabel("Incident Count")
plt.tight_layout()
plt.show()


## 3. Feeder Fault Concentration (Pareto Principle)
We evaluate which feeders account for the disproportionate share of network trips.


In [ ]:
fdr_counts = df_faults["feeder_id"].value_counts().reset_index()
fdr_counts.columns = ["Feeder ID", "Incidents"]

plt.figure(figsize=(11, 4.5))
bars = plt.bar(fdr_counts["Feeder ID"], fdr_counts["Incidents"], color="#dd6b20")
plt.title("Total Unscheduled Breaker Trips by Feeder (6-Month Period)", fontsize=12, fontweight="bold")
plt.xlabel("Feeder ID")
plt.ylabel("Number of Trips")
for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, h + 0.3, str(int(h)), ha="center")
plt.tight_layout()
plt.show()
